In [20]:
from access_parser import AccessParser
import pandas as pd
import re
from pathlib import Path

In [21]:
DATA_DIR = Path('../adept/data')

ASSETS_DIR = DATA_DIR / 'assets'

In [4]:


# .mdb or .accdb file
db = AccessParser("/Users/ben/Downloads/ANGIOontologyTraits_230926.accdb")

In [6]:
table = db.parse_table("angiosperm_traits")

In [10]:
df = pd.DataFrame(table)

In [12]:
df.columns

Index(['sourceDataset', 'termID', 'hasSyn', 'term', 'category', 'trait1',
       'character1', 'Trait 2', 'character 2', 'Trait 3', 'character 3',
       'Trait 4', 'character 4', 'Trait 5', 'character 5', 'Trait 6',
       'character 6', 'Trait 7', 'character 7', 'Trait 8', 'Character 8',
       'Trait 9', 'Character 9', 'Comments', 'Traits AND/OR Terms',
       'Plants Group', 'French terminology'],
      dtype='object')

In [15]:
# Normalise column names first
df = df.rename(columns=lambda c: re.sub(
    r'\s+',
    '',
    str(c).strip().lower()
))

# columns like character1, character2...
character_cols = {
    int(re.search(r'\d+', c).group()): c
    for c in df.columns
    if re.fullmatch(r'character\d+', c)
}

# columns like trait1, trait2...
trait_cols = {
    int(re.search(r'\d+', c).group()): c
    for c in df.columns
    if re.fullmatch(r'trait\d+', c)
}

# Columns that aren't character/trait columns
id_cols = [
    c for c in df.columns
    if c not in character_cols.values()
    and c not in trait_cols.values()
]

rows = []

for i in sorted(set(character_cols) | set(trait_cols)):
    tmp = df[id_cols].copy()

    tmp["character"] = (
        df[character_cols[i]]
        if i in character_cols
        else pd.NA
    )

    tmp["trait"] = (
        df[trait_cols[i]]
        if i in trait_cols
        else pd.NA
    )

    rows.append(tmp)

long_df = pd.concat(rows, ignore_index=True)

# Remove empty character/trait combinations
long_df = long_df.dropna(
    subset=["character", "trait"],
    how="all"
)

In [18]:
new_df = long_df[["term", "category", "character", "trait"]]

In [22]:
old = pd.read_parquet(ASSETS_DIR / 'traits.parquet')

In [27]:
old[~((old.term.isin(new_df.term)) & (old.character.isin(new_df.character)))]

,term,category,character,trait,part,type,group
0,v-form,arrangement,conduplicate,leaf architecture,leaf,discrete,angiosperm
20,aerial-rooted,architecture,climber/scrambler,habit,None,discrete,angiosperm
46,awl-shaped,shape,aciculate,leaf shape,leaf,discrete,angiosperm
52,berry-like,architecture,berry,fruit type,fruit,discrete,angiosperm
85,cordate-ovate,shape,ovate,leaf shape,leaf,discrete,angiosperm
...,...,...,...,...,...,...,...
3625,black-girdled,coloration,black,pollen colour,pollen,colour,angiosperm
3626,purple-flecked,coloration,purple,pollen colour,pollen,colour,angiosperm
3627,violet-flecked,coloration,purple,pollen colour,pollen,colour,angiosperm
3628,white-flushed,coloration,white,pollen colour,pollen,colour,angiosperm


In [28]:
new_df[~((new_df.term.isin(old.term)) & (new_df.character.isin(old.character)))]

,term,category,character,trait
0,v_form,arrangement,conduplicate,Leaf architecture
20,aerial_rooted,architecture,climber/scrambler,Habit
46,awl_shaped,shape,aciculate,Leaf shape
52,berry_like,architecture,berry,Fruit type
58,branching*,architecture,None,Habit
...,...,...,...,...
15328,black_girdled,coloration,black,Petal colour
15329,purple_flecked,coloration,purple,Petal colour
15330,violet_flecked,coloration,purple,Petal colour
15331,white_flushed,coloration,white,Petal colour
